# rHEALPix compact

Step-by-step animation of [`rhealpix_compact`](https://github.com/opengeoshub/vgrid/blob/main/vgrid/conversion/dggscompact/rhealpixcompact.py) / `rhealpixcompact`.

Input: [`rhealpix_10.geojson`](rhealpix_10.geojson) in this folder.

The compact algorithm repeatedly groups cells by parent ID; when all children of a parent are present, they are replaced by the parent cell:

1. Load unique `rhealpix` tokens from the GeoJSON
2. Loop: group by `parent = id[:-1]`, compare children to `parent_cell.subcells()`
3. Merge complete sibling sets into the parent until no more changes (typically **9 children → 1 parent**)
4. Build output geometries with `rhealpix2geo`

**1** shows the full input grid. **3a** frames highlight each merge: cyan children and orange **parent** (dashed outline).

## Install necessary packages

In [ ]:
%pip install vgrid geopandas matplotlib imageio pillow
# optional for MP4:
%pip install imageio-ffmpeg

In [1]:
"""Step-by-step rhealpix_compact animation."""
from collections import defaultdict
from pathlib import Path

import geopandas as gpd
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon

from vgrid.conversion.dggs2geo.rhealpix2geo import rhealpix2geo
from vgrid.conversion.dggscompact.rhealpixcompact import rhealpix_compact
from vgrid.dggs.rhealpixdggs.dggs import WGS84_003 as rhealpix_dggs


INPUT_GEOJSON = Path("rhealpix_11.geojson")
RHEALPIX_ID_FIELD = "rhealpix"
OUT_GIF = "rhealpix_compact.gif"
OUT_MP4 = "rhealpix_compact.mp4"
FRAME_EVERY_MERGE = 1  # 1 = every merge group; use 2+ to skip some groups
DPI = 120
FIX_ANTIMERIDIAN = None


def cell_patches(cell_polys, facecolor, edgecolor, alpha=0.85, lw=1.0):
    patches = []
    for poly in cell_polys:
        if poly is None or poly.is_empty:
            continue
        patches.append(MplPolygon(list(poly.exterior.coords), closed=True))
    return PatchCollection(
        patches,
        facecolor=facecolor,
        edgecolor=edgecolor,
        alpha=alpha,
        linewidths=lw,
        zorder=2,
    )


def polys_for_ids(cell_ids, fix_antimeridian=None):
    polys = []
    for cell_id in cell_ids:
        poly = rhealpix2geo(cell_id, fix_antimeridian=fix_antimeridian)
        if poly is not None and not poly.is_empty:
            polys.append(poly)
    return polys


def render_frame(
    bounds,
    title,
    path,
    background_polys=None,
    child_polys=None,
    parent_poly=None,
):
    fig, ax = plt.subplots(figsize=(8, 8))
    minx, miny, maxx, maxy = bounds
    pad = max(maxx - minx, maxy - miny) * 0.06 or 0.01
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)

    if background_polys:
        ax.add_collection(
            cell_patches(background_polys, "#e8eaf6", "#5c6bc0", alpha=0.5, lw=0.8)
        )
    if child_polys:
        ax.add_collection(
            cell_patches(child_polys, "#00bcd4", "#006064", alpha=0.9, lw=1.4)
        )
    if parent_poly is not None:
        ax.add_collection(
            cell_patches([parent_poly], "#ff9800", "#e65100", alpha=0.55, lw=2.0)
        )
        gpd.GeoSeries([parent_poly.boundary]).plot(
            ax=ax, color="#e65100", lw=3.5, linestyle="--", zorder=4
        )

    ax.set_facecolor("#fafafa")
    ax.plot([], [], color="#00bcd4", lw=4, label="9 children (merge group)")
    ax.plot([], [], color="#ff9800", lw=4, label="parent cell")
    ax.plot([], [], color="#5c6bc0", lw=4, label="other cells")
    ax.legend(loc="upper right", fontsize=8, framealpha=0.95)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(False)
    fig.subplots_adjust(left=0.08, right=0.92, top=0.92, bottom=0.08)
    fig.savefig(path, dpi=DPI, facecolor="white")
    plt.close(fig)


def find_merge_groups(current_ids):
    """Return [(parent_id, [child_ids, ...]), ...] for one compact pass."""
    grouped = defaultdict(set)
    for rhealpix_id in current_ids:
        if len(rhealpix_id) > 1:
            grouped[rhealpix_id[:-1]].add(rhealpix_id)
    merges = []
    for parent, children in grouped.items():
        parent_uids = (parent[0],) + tuple(map(int, parent[1:]))
        parent_cell = rhealpix_dggs.cell(parent_uids)
        subcells = set(str(subcell) for subcell in parent_cell.subcells())
        if children == subcells:
            merges.append((parent, sorted(children)))
    return sorted(merges, key=lambda m: m[0])


def apply_merges(current_ids, merges):
    new_ids = set(current_ids)
    for parent, children in merges:
        new_ids.difference_update(children)
        new_ids.add(parent)
    return new_ids


def rhealpix_compact_with_frames(rhealpix_ids, bounds, frame_dir, fix_antimeridian=None):
    frame_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    idx = 0
    id_to_poly = {}

    def poly_for(cell_id):
        if cell_id not in id_to_poly:
            id_to_poly[cell_id] = rhealpix2geo(cell_id, fix_antimeridian=fix_antimeridian)
        return id_to_poly[cell_id]

    def snap_simple(polys, title):
        nonlocal idx
        p = frame_dir / f"frame_{idx:04d}.png"
        render_frame(bounds, title, p, background_polys=polys)
        frames.append(p)
        idx += 1

    def snap_merge(current_ids, parent_id, child_ids, title):
        nonlocal idx
        child_set = set(child_ids)
        background = [
            poly_for(cid)
            for cid in current_ids
            if cid not in child_set
            and poly_for(cid) is not None
            and not poly_for(cid).is_empty
        ]
        children = [
            poly_for(cid)
            for cid in child_ids
            if poly_for(cid) is not None and not poly_for(cid).is_empty
        ]
        parent_poly = poly_for(parent_id)
        p = frame_dir / f"frame_{idx:04d}.png"
        render_frame(
            bounds,
            title,
            p,
            background_polys=background,
            child_polys=children,
            parent_poly=parent_poly,
        )
        frames.append(p)
        idx += 1

    input_polys = [
        p
        for cid in rhealpix_ids
        for p in [poly_for(cid)]
        if p is not None and not p.is_empty
    ]
    snap_simple(
        input_polys,
        f"1. Input grid ({len(rhealpix_ids)} cells)",
    )

    current = set(rhealpix_ids)
    round_idx = 0
    while True:
        merges = find_merge_groups(current)
        if not merges:
            break
        round_idx += 1
        for merge_i, (parent_id, child_ids) in enumerate(merges, start=1):
            if merge_i % FRAME_EVERY_MERGE != 0 and merge_i != len(merges):
                continue
            n_children = len(child_ids)
            snap_merge(
                current,
                parent_id,
                child_ids,
                f"3a. Round {round_idx} merge {merge_i}/{len(merges)}: "
                f"{n_children} children → parent {parent_id}",
            )
        current = apply_merges(current, merges)
        round_polys = polys_for_ids(sorted(current), fix_antimeridian)
        snap_simple(
            round_polys,
            f"3b. After round {round_idx}: {len(current)} cells",
        )

    final_ids = sorted(rhealpix_compact(rhealpix_ids))
    final_polys = polys_for_ids(final_ids, fix_antimeridian)
    snap_simple(
        final_polys,
        f"4. Final compact set ({len(final_ids)} cells)",
    )
    return frames, final_ids


def main():
    print(f"Using {INPUT_GEOJSON}")
    gdf = gpd.read_file(INPUT_GEOJSON)
    rhealpix_ids = sorted(gdf[RHEALPIX_ID_FIELD].drop_duplicates().tolist())
    if not rhealpix_ids:
        raise ValueError(f"No '{RHEALPIX_ID_FIELD}' values in {INPUT_GEOJSON}")

    bounds = gdf.total_bounds  # minx, miny, maxx, maxy
    frame_dir = Path("_rhealpix_compact_frames")
    frames, final_ids = rhealpix_compact_with_frames(
        rhealpix_ids, bounds, frame_dir, fix_antimeridian=FIX_ANTIMERIDIAN
    )
    GIF_FRAME_DURATION = 1.8  # seconds per frame (slower playback)
    imageio.mimsave(
        OUT_GIF, [imageio.imread(f) for f in frames], duration=GIF_FRAME_DURATION
    )
    print(
        f"Wrote {OUT_GIF} ({len(frames)} frames, "
        f"{len(rhealpix_ids)} → {len(final_ids)} cells)"
    )
    try:
        MP4_FPS = 2.4  # faster playback than default ~1.2
        writer = imageio.get_writer(OUT_MP4, fps=MP4_FPS)
        for f in frames:
            writer.append_data(imageio.imread(f))
        writer.close()
        print(f"Wrote {OUT_MP4}")
    except Exception as e:
        print(f"MP4 skipped ({e}). GIF is enough.")


if __name__ == "__main__":
    main()

Using rhealpix_11.geojson
Wrote rhealpix_compact.gif (202 frames, 1972 → 388 cells)
Wrote rhealpix_compact.mp4
